# NewsAPI

## Set Up

In [64]:
import os
from dotenv import load_dotenv, dotenv_values
from pathlib import Path

In [65]:
load_dotenv(Path("/Users/tanaphan/Documents/STUDY/UoBristol/Final_Project/team_21_lloyds/.env"))

True

In [66]:
API_KEY_NEWS = os.getenv("API_KEY_NEWS")

## EDA

In [67]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#https://newsapi.org/v2/everything
#https://newsapi.org/v2/top-headlines

# FUNCTION: Pull NewsAPI - Everything Endpoint
def pull_news(KEYWORD):
    response = requests.get("https://newsapi.org/v2/everything", params={
        "q": f'"{KEYWORD}"', # Search using this Keyword
        "language": "en",
        "sortBy": "publishedAt", # relevancy, popularity, publishedAt
        "pageSize": 100, # 100 Max
        "page": 1,
        "apiKey": API_KEY_NEWS,
    })
    data = response.json()
    articles = data.get("articles", [])
    print(f"API status     : {data.get('status')}")
    print(f"Total results: {data.get('totalResults')}")
    print(f"Total articles pulled: {len(articles)}")
    if data.get("status") == "error":
        print(f"Error code   : {data.get('code')}")
        print(f"Error message: {data.get('message')}")

    df = pd.json_normalize(articles)
    return df

In [68]:
# Set Keyword
KEYWORD = "lidl"

print("="*15)
print(f"News for {KEYWORD}")
print("="*15)

df = pull_news(KEYWORD)

# Shape
print(f"Shape: {df.shape}")
print(f"\nColumns and types:")
print(df.dtypes)

News for lidl
API status     : ok
Total results: 182
Total articles pulled: 94
Shape: (94, 9)

Columns and types:
author         object
title          object
description    object
url            object
urlToImage     object
publishedAt    object
content        object
source.id      object
source.name    object
dtype: object


In [69]:
print(df.head)

<bound method NDFrame.head of                                     author  \
0                       The Journal reader   
1                Casey L. Moore, USA TODAY   
2   Adam Millington - BBC Sport journalist   
3                           The Inner Ring   
4       Ben Collins - BBC Sport journalist   
..                                     ...   
89                         Olivia Kelleher   
90                        Natalie Culleton   
91                           Kendall Waltz   
92                           Dennis Romboy   
93                       Ottoline Spearman   

                                                title  \
0   Money Diaries: A charity policy coordinator on...   
1   Tour de France Stage 9 winner: Mathieu van der...   
2   Van der Poel wins shortened Tour stage in 40C ...   
3                      Tour de France Stage 9 Preview   
4             Merlier makes it back-to-back Tour wins   
..                                                ...   
89  Man suffered '

In [70]:
# Missing Values
print("Missing Value:")
print(df.isnull().sum())

# Duplication
print("\nDuplicatation:")
dup_title = df.duplicated(subset=["title"]).sum()
dup_description = df.duplicated(subset=["description"]).sum()
dup_url   = df.duplicated(subset=["url"]).sum()
print(f"Duplicate titles : {dup_title}")
print(f"Duplicate description : {dup_description}")
print(f"Duplicate URLs   : {dup_url}")

Missing Value:
author          4
title           0
description     0
url             0
urlToImage      1
publishedAt     0
content         0
source.id      66
source.name     0
dtype: int64

Duplicatation:
Duplicate titles : 2
Duplicate description : 2
Duplicate URLs   : 0


In [71]:
# Articles overtime
print("Article Timeline")
df["publishedAt"] = pd.to_datetime(df["publishedAt"])
df["date"] = df["publishedAt"].dt.date # create a column "date"
daily_counts = df.groupby("date").size().reset_index(name="count")
print(daily_counts)
plt.figure(figsize=(len(daily_counts) * 0.7, 4))
plt.plot(daily_counts["date"], daily_counts["count"], marker="o", color="blue")
plt.fill_between(daily_counts["date"], daily_counts["count"], alpha=0.1, color="blue")
plt.title(f"Articles per day | Keyword = {KEYWORD}")
plt.xlabel("Date")
plt.ylabel("Article Count")
plt.savefig(f"newsapi_{KEYWORD}_timeline.png") # Save graph
plt.close()

# Source Distribution
print("\nSource Distribution")
source_counts = df.groupby("source.name").size().reset_index(name="count").sort_values("count", ascending=False)
print(source_counts)
plt.figure(figsize=(10, len(source_counts) * 0.4))
plt.barh(source_counts["source.name"], source_counts["count"], color="green")
plt.title(f"Articles by source | Keyword = {KEYWORD}")
plt.xlabel("Article count")
plt.savefig(f"newsapi_{KEYWORD}_sources.png", bbox_inches="tight")
plt.close()

# Text Length
df["title_len"] = df["title"].fillna("").apply(len)
df["description_len"] = df["description"].fillna("").apply(len)

# Save as .CSV file
output_cols = ["date", "source.name", "title", "description", "content",
               "title_len", "description_len"]
df[output_cols].to_csv(f"newsapi_{KEYWORD}.csv", index=False)

Article Timeline
          date  count
0   2026-07-01      6
1   2026-07-02      9
2   2026-07-03     14
3   2026-07-04     11
4   2026-07-05      5
5   2026-07-06      9
6   2026-07-07     12
7   2026-07-08      6
8   2026-07-09      6
9   2026-07-10      8
10  2026-07-11      5
11  2026-07-12      3

Source Distribution
                   source.name  count
8                Dailymail.com     13
1                     BBC News     12
15                   Inrng.com     11
28                   USA Today      9
23             The Irish Times      7
19                         RTE      4
17             PezCycling News      4
32                     road.cc      3
26               TheJournal.ie      3
29         Yahoo Entertainment      2
24           The Local Germany      2
0                ABC News (AU)      2
4                          CNA      2
12                Flobikes.com      1
6                          CNN      1
31          disneyfoodblog.com      1
30                 allears.net

## Sentiment Analysis

In [72]:
!pip install vaderSentiment

In [73]:
df = pd.read_csv("newsapi_lidl.csv")

print(f"Shape: {df.shape}")
print(f"Column Names: {list(df.columns)}")

Shape: (94, 7)
Column Names: ['date', 'source.name', 'title', 'description', 'content', 'title_len', 'description_len']


In [74]:
# Drop Duplications
df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["description"]).reset_index(drop=True)
print(f"Shape: {df.shape}")

Shape: (92, 7)


In [75]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_score(df):
    sentences = df["title"].fillna("") + " " + df["description"].fillna("")
    return sentences.apply(analyzer.polarity_scores)

df["scores"] = get_score(df)
df["compound_scores"] = df["scores"].str["compound"]

df["sentiment"] = df["compound_scores"].apply(
    lambda s: "positive" if s >= 0.5 else
              "negative" if s <= -0.5 else
              "neutral"
)

df[["title", "compound_scores", "sentiment"]]

# df.to_csv(f"sentiment_analysis.csv", index=False)


,title,compound_scores,sentiment
0,Money Diaries: A charity policy coordinator on...,0.4215,neutral
1,Tour de France Stage 9 winner: Mathieu van der...,0.8074,positive
2,Van der Poel wins shortened Tour stage in 40C ...,0.8126,positive
3,Tour de France Stage 9 Preview,0.4019,neutral
4,Merlier makes it back-to-back Tour wins,0.8126,positive
...,...,...,...
87,Man suffered 'violent death' and was found wit...,-0.9661,negative
88,3 Disney World Essentials Drop At LIDL This Week,-0.3382,neutral
89,Lidl Just Dropped NEW Disney World Travel Esse...,0.0000,neutral
90,These 6 Americans will compete in the Tour de ...,0.0000,neutral


In [76]:
# Features to be trained
features = {
    "company_name": "Lidl",
    "article_count": len(df),
    "avg_compound_score": round(df["compound_scores"].mean(), 3),
    "pct_positive": round((df["sentiment"] == "positive").mean(), 3),
    "pct_negative": round((df["sentiment"] == "negative").mean(), 3),
    "pct_neutral": round((df["sentiment"] == "neutral").mean(), 3),
}

pd.DataFrame([features]).to_csv("newsapi_features_lidl.csv", index=False)